In [238]:
import pandas as pd
import numpy as np
# import xlsxwriter as xl


In [239]:
df1=pd.read_excel('product.xlsx')
df2=pd.read_excel('sales.xlsx')

In [240]:
df=pd.merge(df2,df1,on='Product ID',how='left')

In [241]:
df['Date']=pd.to_datetime(df['Date']).dt.date

In [242]:
df['Product_Missing']=df[['Product_Name','Product_Cost']].isnull().any(axis=1)
# .any(axis=1) tells to check if any of the two is missing trough column, while all(axis=1) check if both are missing

In [243]:
df['Employee']=df['Employee'].fillna('Not confirmed')

In [244]:
df['Revenue']=np.nan #
valid_mask=(df['Product_Missing']==False)  & (df['quantity_present']==True)
df.loc[valid_mask,'Revenue']=df.loc[valid_mask,'Quantity']*df.loc[valid_mask,'Selling_Price']

In [245]:
df['Cost']=np.nan
df.loc[valid_mask,'Cost']=df.loc[valid_mask,'Quantity']*df.loc[valid_mask,'Product_Cost']

In [246]:
df['Profit']=np.nan
df.loc[valid_mask,'Profit']=df.loc[valid_mask,'Revenue']-df.loc[valid_mask,'Cost']

In [247]:
df=df.rename(columns={"quantity_present":"Qty_Present",'Product_Cost':'Prod_Cost','Product_Name':'Prod_Name','Product_Missing':'Prod_Missed'})

In [248]:
df['Sales_status']='Broken'

df.loc[df['Quantity']==0,'Sales_status']='Zero_Impact'

check_mask=(df['Revenue'].notna()) & (df['Cost'].notna()) & (df['Profit'].notna())
df.loc[check_mask,'Sales_status']='Valid'

In [249]:
df.head()

,Date,Product ID,Quantity,Employee,Region,Qty_Present,Prod_Name,Category,Prod_Cost,Selling_Price,Stock,Prod_Missed,Revenue,Cost,Profit,Sales_status
0,2025-12-24,P-05,0,Ali,North,False,Folder,Office,30.0,45.0,60.0,False,NaN,NaN,NaN,Zero_Impact
1,2025-12-07,P-02,2,Ali,North,True,Notebook,Stationary,50.0,70.0,40.0,False,140.0,100.0,40.0,Valid
2,2025-12-23,P-02,1,Ali,North,True,Notebook,Stationary,50.0,70.0,40.0,False,70.0,50.0,20.0,Valid
3,2025-12-08,P-02,2,Ali,South,True,Notebook,Stationary,50.0,70.0,40.0,False,140.0,100.0,40.0,Valid
4,2025-12-27,P-02,2,Sara,North,True,Notebook,Stationary,50.0,70.0,40.0,False,140.0,100.0,40.0,Valid


In [250]:
total_mask=df['Sales_status']=='Valid'
Total_revenue=df.loc[total_mask,'Revenue'].sum()

Total_Cost=df.loc[total_mask,'Cost'].sum()

Total_Profit=df.loc[total_mask,'Profit'].sum()

In [251]:
Total_sales=df.groupby('Sales_status').size()
Total_sales=Total_sales.reset_index(name='Total_number')


In [252]:
Profit_BY_Product=df.groupby('Prod_Name')['Profit'].sum()
Profit_BY_Product=Profit_BY_Product.reset_index()

Revenue_by_Product=df.groupby('Prod_Name')['Revenue'].sum()
Revenue_by_Product=Revenue_by_Product.reset_index()

In [253]:
Sale_by_product=df.groupby('Prod_Name')['Sales_status'].value_counts()
Sale_by_product=Sale_by_product.reset_index()

In [254]:
Summary={
    'Total_Revenue':Total_revenue,
    'Total_Cost':Total_Cost,
    'Total_Profit':Total_Profit}

summary=pd.DataFrame([Summary])


In [255]:
writer=pd.ExcelWriter('REPORT.xlsx', engine='xlsxwriter')
wb=writer.book

In [256]:
formats = {
     'header': wb.add_format({'bold':True,'align':'center','bg_color':'#2F75B6','font_color':'white'}),
      'currency':wb.add_format({'num_format': '#,##0.00'}),
       'Values':wb.add_format({'align':'center','font_name':'Arial'}),
        'totals':wb.add_format({'top':2,'bold':True}),
         'currency_totals':wb.add_format({'num_format': '#,##0.00','top':2,'bold':True}),
        'low_profit': wb.add_format({'bg_color':'#FFC7CE','font_color':'#9C0006'}),
         'high_profit': wb.add_format({'bg_color':'#C6EFCE','font_color':'#276221'}),

}

In [257]:


df.to_excel(writer,sheet_name='Data',index=False),
summary.to_excel(writer,sheet_name='Summary',index=False)
# Sale_by_product.to_excel(writer,sheet_name='Sale_by_product',index=False)
Revenue_by_Product.to_excel(writer,sheet_name='Revenue_by_Product',index=False)
Profit_BY_Product.to_excel(writer,sheet_name='Profit_by_product',index=False)
Total_sales.to_excel(writer,sheet_name='Total_sales',index=False)





In [258]:
ws = writer.sheets['Data']

# Column widths (keep yours if good, or improve)
ws.set_column(0, len(df.columns)-1, 10)
ws.set_column(0, 0, 18)   # Date
ws.set_column(3, 3, 18)   # Employee
ws.set_column(6, 6, 20)   # Prod_Name
ws.set_column(12,14, 14)  # Revenue, Cost, Profit wider
# -------------------------------------#



# FOR HEADER FORMAT AT EACH
for col_value,index in enumerate(df.columns):
 ws.write(0,col_value,index,formats['header'])
# -------------------------------------#

ws.set_column(8, 8, 10, formats['currency'])
ws.set_column(9, 9, 10, formats['currency'])
ws.set_column(13, 13, 10, formats['currency'])
# ws.set_column(13, 13, 10, formats['currency'])


# FREEZE HEADERS AT A PLACE
ws.freeze_panes(1,0)

# FOR ZEBRA PATTERN
even_row=wb.add_format({'bg_color':"#F7F9FC"})
odd_row=wb.add_format({'bg_color':"#E9EEF5"})
for r in range(1,len(df)+1):
 fmt=even_row if r % 2==0 else odd_row
 ws.set_row(r,cell_format=fmt)





In [259]:
ws_Summary=writer.sheets['Summary']
ws_Summary.set_column(0,len(summary.columns)-1,10)
ws_Summary.set_column(0,0,15)
for value,index in enumerate(summary.columns):
    ws_Summary.write(0,value,index,formats['header'])







In [260]:

sale_pivot = Sale_by_product.pivot_table(
    index='Prod_Name',
    columns='Sales_status',
    values='count',
    fill_value=0
).reset_index()

# Write the pivoted data
sale_pivot.to_excel(writer, sheet_name='Sale_by_product', index=False)

# Now get the worksheet
ws_Sale = writer.sheets['Sale_by_product']

# Basic formatting
ws_Sale.set_column(0, 0, 18)  # Product name wider
ws_Sale.set_column(1, len(sale_pivot.columns)-1, 12)

# Header formatting
for col_num, col_name in enumerate(sale_pivot.columns):
    ws_Sale.write(0, col_num, col_name, formats['header'])

# === IMPROVED CHART ===
chart = wb.add_chart({'type': 'column', 'subtype': 'stacked'})

last_row = len(sale_pivot)

for i, status in enumerate(['Valid', 'Zero_Impact']):
    chart.add_series({
        'name':       ['Sale_by_product', 0, i+1],
        'categories': ['Sale_by_product', 1, 0, last_row, 0],
        'values':     ['Sale_by_product', 1, i+1, last_row, i+1],
        'data_labels': {'value': True}
    })

chart.set_title({'name': 'Sales Status by Product'})
chart.set_x_axis({'name': 'Product'})
chart.set_y_axis({'name': 'Number of Transactions'})
chart.set_style(11)

ws_Sale.insert_chart('E3', chart, {'x_offset': 15, 'y_offset': 10, 'x_scale': 1.4, 'y_scale': 1.2})

0

In [261]:
ws_Revenue=writer.sheets['Revenue_by_Product']
ws_Revenue.set_column(0,len(Revenue_by_Product)-1,10)
ws_Revenue.set_column(0,0,15)
for value,col_nam in enumerate(Revenue_by_Product.columns):
    ws_Revenue.write(0,value,col_nam,formats['header'])
# for char_num,char_val in enumerate(Revenue_by_Product):
#     ws_Revenue.write_row(char_num,0,char_val)

chart=wb.add_chart({'type':'column'})
last_row=len(Revenue_by_Product)
chart.add_series({
   'name':'Revenue',
    'categories':['Revenue_by_Product',1,0,last_row,0],
    'values':['Revenue_by_Product',1,1,last_row,1],

'data_labels': {'value': True},           # Show values on top of bars
    'points':[
        {'fill': {'color': '#4472C4'}},   # Folder
        {'fill': {'color': '#ED7D31'}},   # Marker
        {'fill': {'color': '#A5A5A5'}},   # Notebook
        {'fill': {'color': '#FFC000'}} # Pen
    ]
})

chart.set_title({'name':'Revenue_by_Product'})
chart.set_y_axis({'name':'Revenue(PKR)'})
chart.set_x_axis({'name':'Product'})

chart.set_style(11)
ws_Revenue.insert_chart('D3',chart,{'x_offset': 15, 'y_offset': 15, 'x_scale': 1.2, 'y_scale': 1.1})


0

In [262]:
ws_Profit=writer.sheets['Profit_by_product']
ws_Profit.set_column(0,len(Profit_BY_Product)-1,10)
ws_Profit.set_column(0,0,15)
for value,col_nam in enumerate(Profit_BY_Product.columns):
    ws_Profit.write(0,value,col_nam,formats['header'])

profit_charts=wb.add_chart({"type":"column"})
last_row=len(Profit_BY_Product)
profit_charts.add_series({
    'name':'Profit_by_product',
    'categories':['Profit_by_product',1,0,last_row,0],
    'values':['Profit_by_product',1,1,last_row,1],

    'data_labels': {'value': True},
    'points':[
        {'fill': {'color': '#4472C4'}},
        {'fill': {'color': '#ED7D31'}},
        {'fill': {'color': '#A5A5A5'}},
        {'fill': {'color': '#FFC000'}}
]

})
profit_charts.set_title({'name':"Profit_by_Product"}),
profit_charts.set_y_axis({'name':'Profit(PKR)'}),
profit_charts.set_x_axis({'name':'Product'}),

profit_charts.set_style(11)
ws_Profit.insert_chart('D3',profit_charts,{'x_offset': 15, 'y_offset': 15,'x_scale':1.2,'y_scale':1.1})




0

In [263]:

ws_Sales=writer.sheets['Total_sales']
ws_Sales.set_column(0,len(Total_sales)-1,15)
for value,col_nam in enumerate(Total_sales.columns):
    ws_Sales.write(0,value,col_nam,formats['header'])

sales_charts=wb.add_chart({'type':'pie'})
last_row=len(Total_sales)
sales_charts.add_series({
    'name':'Total_sales',
    'categories':['Total_sales',1,0,last_row,0],
    'values':['Total_sales',1,1,last_row,1],
    'data_labels':{'value': True,'percentage':True}

}),
sales_charts.set_title({'name':'Sales_distribution'})

ws_Sales.insert_chart('D4',sales_charts)



0

FOR LOW SROCK ALERT


In [264]:
avg_stock=df['Stock'].mean().round(2)
low_stock_mask=df['Stock']<=avg_stock
low_stock_df = df.loc[low_stock_mask, :]
low_stock_df.to_excel(writer,sheet_name='Low_stocks',index=False)
ws_low_stock=writer.sheets['Low_stocks']
ws_low_stock.set_column(0,len(low_stock_df)-1,15)
for value,col_nam in enumerate(low_stock_df.columns):
    ws_low_stock.write(0,value,col_nam,formats['header'])



FORMAT FOR LOW  AND HIGH PROFIT DIFFERENCE:


In [265]:
ws.conditional_format(1,14,len(df),14,{
    'type':'cell',
    'criteria':'<=',
    'value':20,
    'format':formats['low_profit']
}),

ws.conditional_format(1,14,len(df),14,{
    'type':'cell',
    'criteria':'>',
    'value':20,
    'format':formats['high_profit']
})

0

FOR DASHBOARD


In [266]:
ws_dashboard=wb.add_worksheet('Dashboard')
ws_dashboard.write(0, 0, 'Total__Revenue', formats['header'])
ws_dashboard.write(0, 1, 'Total_Cost', formats['header'])
ws_dashboard.write(0, 2, 'Total_profit', formats['header'])
ws_dashboard.write(0, 3, 'Avg_Stock', formats['header'])
#For values now
ws_dashboard.write(1, 0,Total_revenue,formats[ 'currency_totals']),
ws_dashboard.write(1, 1, Total_Cost, formats[ 'currency_totals']),
ws_dashboard.write(1, 2, Total_Profit, formats[ 'currency_totals']),
ws_dashboard.write(1, 3, avg_stock, formats[ 'Values'])

#FOR WIDTH:
ws_dashboard.set_column(0,3,15)
# first sheet
ws_dashboard.set_first_sheet()
ws_dashboard.activate()


In [267]:
ws.add_table(0,0,len(df),len(df.columns)-1,{'columns':[{'header':col} for col in df.columns]
,'style':'Table Style Medium 9'})
# fix some width
ws.set_column(1,1,15)
ws.set_column(5,5,18)
ws.set_column(8,8,18)
ws.set_column(9,9,18)
ws.set_column(11,11,18)
ws.set_column(15,15,18)

0

In [268]:
writer.close()